# Charging a capacitor (RC circuit)

*em · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/rc_circuit)

A capacitor charging through a resistor obeys $dV/dt = (V_s - V)/(RC)$. Several circuits with different resistances are stacked into one tensor of voltages, so a single Runge–Kutta step advances all of them. Switch the source to a square wave to watch the capacitors charge and discharge.

**Vector form**

$$V_s = R\,I + V,\qquad I = C\,\frac{dV}{dt}\;\;\Rightarrow\;\;\frac{dV}{dt} = \frac{V_s - V}{RC}$$

**Index (tensor) form**

$$\dot V_b = \frac{V_s - V_b}{R_b\,C},\qquad I_b = \frac{V_s - V_b}{R_b},\qquad U_b = \tfrac12\,C\,V_b^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_b$ | `V` | (B,) | voltage across capacitor $b$ (one entry per circuit) |
| $R_b$ | `R` | (B,) | resistance of circuit $b$ |
| $C$ | `C` | scalar | capacitance, the same in every circuit |
| $V_s(t)$ | `source(t, V0, shape, period)` | scalar | source voltage, shared by all circuits (broadcast over $b$) |
| $\dot V_b$ | `(Vs - V) / (R * C)` | (B,) | element-wise: index $b$ appears once on each side, nothing is summed |
| $I_b$ | `current(V, Vs, R)` | (T, B) | current through each resistor |
| $U_b$ | `stored_energy(V, C)` | (B,) | energy stored in each capacitor |

Units: $R$ in kΩ and $C$ in mF, so $\tau = RC$ comes out in seconds and $I$ in mA. After one time constant the capacitor holds 63 % of the source voltage, after $5\tau$ it is practically full. While charging from a battery the source delivers $CV_s^2$ of energy: exactly half ends up in the capacitor, the other half is turned into heat in $R$ — whatever the value of $R$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/rc_circuit/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import stored_energy


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    C = params["C"]
    R = tensors.resistances(params["R_min"], params["R_max"], params["n_circuits"], dev)  # (B,)
    V0 = tensors.initial_voltage(len(R), dev)  # (B,)

    result = solve.solve(V0, R, C, params["V_s"], params["source"], params["period"], params["t_max"])

    U = stored_energy(result["V"][-1], C)  # (B,)  kΩ·mF = s, V²·mF = mJ
    for r, v, u in zip(R.tolist(), result["V"][-1].tolist(), U.tolist()):
        print(f"R {r:5.2f} kΩ  τ = {r * C:5.2f} s   V(end) = {v:5.2f} V   ½CV² = {u:6.2f} mJ")
    return plot.animate(result, R, C, out_dir / "rc_circuit.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/rc_circuit/equation.py
r"""Physics → tensor translation.

Kirchhoff's voltage law for a resistor R and capacitor C in series with a source V_s(t):

    V_s = R I + V,      I = C \frac{dV}{dt}     ⇒     \frac{dV}{dt} = \frac{V_s - V}{R C}

We solve B circuits at once, one for each resistance R_b. The index b only labels the
circuit — nothing is summed — so every formula is plain element-wise tensor arithmetic:

    \dot V_b = (V_s - V_b) / (R_b C),     I_b = (V_s - V_b) / R_b,     U_b = ½ C V_b^2

The time constant τ_b = R_b C sets how fast each capacitor charges: V(τ) = (1 − 1/e) V_s.
"""

import torch


def source(t: float, V0: float, shape: str, period: float) -> float:
    """V_s(t): a constant battery ("step"), or a square wave switching between V0 and 0 every half period."""
    if shape == "square" and (t % period) >= period / 2:
        return 0.0
    return V0


def rhs(V: torch.Tensor, Vs: float, R: torch.Tensor, C: float) -> torch.Tensor:
    r"""\dot V_b = (V_s − V_b)/(R_b C).   V, R: (B,) → (B,)"""
    return (Vs - V) / (R * C)


def current(V: torch.Tensor, Vs: torch.Tensor, R: torch.Tensor) -> torch.Tensor:
    """I_b = (V_s − V_b)/R_b: the voltage left over for the resistor, divided by R.   → same shape as V"""
    return (Vs - V) / R


def stored_energy(V: torch.Tensor, C: float) -> torch.Tensor:
    """U_b = ½ C V_b²: energy in the capacitor's electric field.   → same shape as V"""
    return 0.5 * C * V**2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/rc_circuit/tensors.py
"""Build the concrete tensors: one resistance per circuit and the starting voltages."""

import torch


def resistances(R_min: float, R_max: float, n: int, device: torch.device) -> torch.Tensor:
    """R_b of shape (B,): n resistances (kΩ) evenly spread from R_min to R_max."""
    if n == 1:
        return torch.tensor([R_min], device=device)
    return torch.linspace(R_min, R_max, n, device=device)


def initial_voltage(n: int, device: torch.device) -> torch.Tensor:
    """V_b(0) of shape (B,): every capacitor starts empty."""
    return torch.zeros(n, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/rc_circuit/solve.py
"""Step all B circuits forward together with 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import current, rhs, source


def solve(V0, R, C: float, Vs0: float, shape: str, period: float, t_max: float, dt: float = 0.01) -> dict:
    """Return a dict of tensors: t (T,), Vs (T,), V (T, B), I (T, B).

    One RK4 step updates the voltage of every circuit at once: V is a tensor of shape (B,).
    """
    n_steps = int(round(t_max / dt))
    V = integrate(lambda V, t: rhs(V, source(t, Vs0, shape, period), R, C), V0, dt, n_steps)  # (T, B)
    t = torch.arange(n_steps + 1, device=V0.device) * dt  # (T,)
    Vs = torch.tensor([source(s, Vs0, shape, period) for s in t.tolist()], device=V0.device)  # (T,)
    I = current(V, Vs[:, None], R)  # (T, B)   Vs broadcast over the circuits b  # noqa: E741
    return {"t": t, "Vs": Vs, "V": V, "I": I}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/rc_circuit/plot.py
"""Animate the circuits: a bar per capacitor voltage, and the V(t) and I(t) curves growing in time."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(result: dict, R: torch.Tensor, C: float, path: Path) -> Path:
    t = result["t"].cpu().numpy()  # (T,)
    Vs = result["Vs"].cpu().numpy()  # (T,)
    V = result["V"].cpu().numpy()  # (T, B)
    I = result["I"].cpu().numpy()  # (T, B)  # noqa: E741
    labels = [f"{r:.2g}\nτ={r * C:.2g}s" for r in R.tolist()]  # bar labels: resistance and time constant
    top = max(Vs.max(), 1e-6) * 1.1

    fig = plt.figure(figsize=(7, 4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.6])
    ax_bar = fig.add_subplot(grid[:, 0])
    ax_V = fig.add_subplot(grid[0, 1])
    ax_I = fig.add_subplot(grid[1, 1], sharex=ax_V)

    # left: one bar per capacitor
    colours = [f"C{b}" for b in range(V.shape[1])]
    bars = ax_bar.bar(range(V.shape[1]), V[0], color=colours)
    source_line = ax_bar.axhline(Vs[0], color="k", ls="--", lw=1, label="source V_s")
    ax_bar.set_xticks(range(V.shape[1]), labels, fontsize=7)
    ax_bar.set_xlabel("R (kΩ)")
    ax_bar.set_ylabel("capacitor voltage V (V)")
    ax_bar.set_ylim(0, top)
    ax_bar.grid(axis="x", visible=False)
    ax_bar.legend(loc="upper right", fontsize=8)
    ax_bar.set_title("t = 0.0 s")  # placeholder so tight_layout leaves room for the clock

    # right: V(t) and I(t) for every circuit
    lines_V = [ax_V.plot([], [], color=c, lw=1.5)[0] for c in colours]
    lines_I = [ax_I.plot([], [], color=c, lw=1.5)[0] for c in colours]
    (source_curve,) = ax_V.plot([], [], color="k", ls="--", lw=1)
    ax_V.set_xlim(0, t[-1])
    ax_V.set_ylim(0, top)
    ax_V.set_ylabel("V (V)")
    ax_I.set_ylim(min(I.min(), 0) * 1.1, max(I.max(), 1e-6) * 1.1)
    ax_I.set_ylabel("I (mA)")
    ax_I.set_xlabel("t (s)")
    plt.setp(ax_V.get_xticklabels(), visible=False)

    def update(step: int):
        for b, bar in enumerate(bars):
            bar.set_height(V[step, b])
            lines_V[b].set_data(t[: step + 1], V[: step + 1, b])
            lines_I[b].set_data(t[: step + 1], I[: step + 1, b])
        source_line.set_ydata([Vs[step]] * 2)
        source_curve.set_data(t[: step + 1], Vs[: step + 1])
        ax_bar.set_title(f"t = {t[step]:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'V_s': 5.0,
    'C': 1.0,
    'R_min': 0.5,
    'R_max': 4.0,
    'n_circuits': 4,
    'source': 'step',
    'period': 12.0,
    't_max': 20.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.rc_circuit")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.rc_circuit.main")
problem_dir = Path("problems/em/rc_circuit")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
